Extração de Eventos num periodo passado

In [0]:
# ----------------------------------------------------------------------------------------
# SCRIPT: 1_eventos_ingest
# LOCAL: 1_bronze/src/
# OBJETIVO: Dados brutos referente os eventos extraidos da API da Câmara dos Deputados
# ----------------------------------------------------------------------------------------


import requests as req

data_inicio = "2025-01-01"
data_fim = "2025-05-31"
pagina = 1
todos_eventos = []

while True:
    url = f"https://dadosabertos.camara.leg.br/api/v2/eventos?dataInicio={data_inicio}&dataFim={data_fim}&pagina={pagina}&itens=100&ordem=ASC&ordenarPor=dataHoraInicio"
    response = req.get(url).json()
    dados = response['dados']
    
    if not dados:
        break
        
    todos_eventos.extend(dados)
    pagina += 1

df_eventos = spark.createDataFrame(todos_eventos)
df_eventos.write.mode("overwrite").saveAsTable("bronze_eventos")
print(f"Total baixado: {len(todos_eventos)} eventos em {pagina-1} páginas.")

In [0]:
display(df_eventos.orderBy("dataHoraInicio"))

Extração de eventos futuros

In [0]:
# ----------------------------------------------------------------------------------------
# SCRIPT: 1_eventos_ingest
# LOCAL: 1_bronze/src/
# OBJETIVO: Foco em eventos com datas posteriores a hoje
# ----------------------------------------------------------------------------------------


import requests as req
from datetime import date

# Configurações Iniciais
data_hoje = date.today().strftime('%Y-%m-%d')
data_fim = "2026-07-31"
pagina = 1
todos_eventos = []

print(f"Iniciando captura de eventos futuros a partir de {data_hoje}...")

# Loop de Paginação
while True:
    url = f"https://dadosabertos.camara.leg.br/api/v2/eventos?dataInicio={data_hoje}&dataFim={data_fim}&pagina={pagina}&itens=100&ordem=ASC&ordenarPor=dataHoraInicio"
    
    try:
        res = req.get(url).json()
        dados_da_pagina = res['dados']

        # Se página vazia, interrompe o loop
        if not dados_da_pagina:
            break
            
        todos_eventos.extend(dados_da_pagina)
        print(f"Página {pagina} capturada com {len(dados_da_pagina)} eventos...")
        
        pagina += 1
        
    except Exception as e:
        print(f"Erro na captura da página {pagina}: {e}")
        break

# Salvar na Bronze
if todos_eventos:
    # Uso do esquema da tabela de eventos existente para evitar erros de inferência de tipos nulos
    esquema_referencia = spark.table("bronze_eventos").schema
    
    df_futuros = spark.createDataFrame(todos_eventos, schema=esquema_referencia)
    
    df_futuros.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze_eventos_futuros")
    
    print(f"Sucesso! Total de {len(todos_eventos)} eventos salvos.")
else:
    print("Nenhum evento futuro encontrado.")

display(spark.table("bronze_eventos_futuros"))